In [0]:
# 1. 声明原始文件在 Volume 里的路径
products_csv_path = "/Volumes/workspace/default/olist_files/olist_products_dataset.csv"
translation_csv_path = "/Volumes/workspace/default/olist_files/product_category_name_translation.csv"

In [0]:
# 2. 分布式读取 CSV 文件
# .option("inferSchema", "true") 会让 Spark 分布式扫描一遍数据，自动判定字段是 String 还是 Integer
raw_products_df = spark.read.format("csv") \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .load(products_csv_path)

raw_translation_df = spark.read.format("csv") \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .load(translation_csv_path)

In [0]:
raw_products_df.printSchema()

In [0]:
display(raw_products_df.limit(5))

In [0]:
raw_products_df.show(5)

In [0]:
total_products = raw_products_df.count()
print(f"There are {total_products:,} products in the raw_products_df")

In [0]:
# 1. 挑选需要的核心字段
cropped_df = raw_products_df.select(
    "product_id", 
    "product_category_name", 
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
)

# 看看裁剪后的表骨架，多余的字段是不是全消失了
cropped_df.printSchema()

In [0]:
from pyspark.sql import functions as F

In [0]:
# 过滤掉类目为空的脏数据
clean_df = cropped_df.filter(F.col("product_category_name").isNotNull())

# 比对干净表和脏表的行数，看看过滤了多少脏数据
total_clean_products = clean_df.count()
total_dirty_products = cropped_df.count() - total_clean_products
print(f"There are {total_clean_products:,} products in the clean_df")
print(f"There are {total_dirty_products:,} products in the cropped_df")

In [0]:
# 3. 链式连招：原地覆盖重量，并衍生体积新列
transformed_df = clean_df \
    .withColumn("product_weight_g", F.col("product_weight_g") / 1000.0) \
    .withColumn("product_volume_cm3", F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm"))

# 验证结果：我们甚至可以在 select 里直接用 .alias() 顺便把列名改得更专业
final_preview_df = transformed_df.select(
    "product_id",
    "product_category_name",
    F.col("product_weight_g").alias("product_weight_kg"), # 改名为 kg 表达更精准
    "product_volume_cm3"
)

display(final_preview_df.limit(5))

In [0]:
# studying aggregation functions

final_preview_df = final_preview_df.groupBy("product_category_name").agg(
    F.avg("product_weight_kg").alias("avg_weight_kg"),
    F.max("product_volume_cm3").alias("max_volume_cm3"),
    F.min("product_volume_cm3").alias("min_volume_cm3"),
    F.count("product_id").alias("total_products")
)
display(final_preview_df.limit(5))